# NASDAQ Delta Table Explorer

Query Delta tables with `%%sql` magic powered by **DuckDB** — no Java, no Spark, no internet.

| View | Contents | Key columns |
|---|---|---|
| `daily`  | Daily OHLCV + indicators | `date`, `symbol`, `Open`, `High`, `Low`, `Close`, `Volume`, `sma_20`, `sma_33`, `ema_20`, `ema_33` |
| `weekly` | Weekly OHLCV + indicators | `week_start`, `symbol`, `Open`, `High`, `Low`, `Close`, `Volume`, `trading_days`, `sma_20`, `sma_33`, `ema_20`, `ema_33` |

## Setup — run once per session

In [8]:
import warnings; warnings.filterwarnings('ignore')
import duckdb, pandas as pd, matplotlib.pyplot as plt, matplotlib.dates as mdates
from IPython.core.magic import register_cell_magic
from IPython import get_ipython

# ── Paths ──────────────────────────────────────────────────────────────────
DAILY_PATH  = 'C:/Users/mohin/Documents/PythonCodes/get_trading_data/data/delta/daily'
WEEKLY_PATH = 'C:/Users/mohin/Documents/PythonCodes/get_trading_data/data/delta/weekly'

# ── DuckDB connection — LOAD (cached, instant) not INSTALL (downloads) ─────
conn = duckdb.connect()
conn.execute('LOAD delta;')
conn.execute(f"CREATE OR REPLACE VIEW daily  AS SELECT * FROM delta_scan('{DAILY_PATH}')")
conn.execute(f"CREATE OR REPLACE VIEW weekly AS SELECT * FROM delta_scan('{WEEKLY_PATH}')")

# ── Lightweight %%sql magic — no jupysql needed ────────────────────────────
@register_cell_magic
def sql(line, cell):
    """%%sql  or  %%sql varname <<  — runs SQL against DuckDB, returns a DataFrame."""
    line = line.strip()
    if '<<' in line:
        varname = line.split('<<')[0].strip()
    else:
        varname = line if line else None
    df = conn.execute(cell).df()
    if varname:
        get_ipython().user_ns[varname] = df
    return df

# ── Display settings ───────────────────────────────────────────────────────
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_rows', 50)
plt.style.use('seaborn-v0_8-darkgrid')

print('Ready  |  kernel: Python 3.14  |  tables: daily, weekly')

Ready  |  kernel: Python 3.14  |  tables: daily, weekly


---
## 1 · Database overview

In [ ]:
%%sql
SELECT 'daily'  AS tbl,
       COUNT(DISTINCT symbol) AS symbols,
       COUNT(*)               AS total_rows,
       MIN(date)::DATE        AS first_date,
       MAX(date)::DATE        AS last_date
FROM daily
UNION ALL
SELECT 'weekly',
       COUNT(DISTINCT symbol),
       COUNT(*),
       MIN(week_start)::DATE,
       MAX(week_start)::DATE
FROM weekly

In [ ]:
%%sql
-- How many symbols have indicators computed?
SELECT
    COUNT(DISTINCT symbol)                                        AS total_symbols,
    COUNT(DISTINCT CASE WHEN sma_20 IS NOT NULL THEN symbol END) AS have_sma20,
    COUNT(DISTINCT CASE WHEN sma_33 IS NOT NULL THEN symbol END) AS have_sma33,
    COUNT(DISTINCT CASE WHEN ema_20 IS NOT NULL THEN symbol END) AS have_ema20,
    COUNT(DISTINCT CASE WHEN ema_33 IS NOT NULL THEN symbol END) AS have_ema33
FROM daily

---
## 2 · Latest candles + indicators for a symbol

In [ ]:
%%sql
SELECT date, symbol, Open, High, Low, Close, Volume,
       ROUND(sma_20,2) AS sma_20, ROUND(sma_33,2) AS sma_33,
       ROUND(ema_20,2) AS ema_20, ROUND(ema_33,2) AS ema_33
FROM   daily
WHERE  symbol = 'AAPL'
ORDER  BY date DESC
LIMIT  20

---
## 3 · Golden cross screen
Symbols where **SMA-20 > SMA-33** (bullish) on the latest trading day

In [ ]:
%%sql
SELECT symbol,
       ROUND(Close,  2) AS Close,
       ROUND(sma_20, 2) AS sma_20,
       ROUND(sma_33, 2) AS sma_33,
       ROUND(ema_20, 2) AS ema_20,
       ROUND(ema_33, 2) AS ema_33
FROM   daily
WHERE  date   = (SELECT MAX(date) FROM daily)
  AND  sma_20 > sma_33
  AND  sma_20 IS NOT NULL
ORDER  BY symbol
LIMIT  50

---
## 4 · MAG7 — normalised price chart

> **Tip:** `%%sql varname <<` assigns the result to a Python variable you can use immediately in the next cell for plotting.

In [ ]:
%%sql mag7 <<
SELECT date, symbol, Close
FROM   daily
WHERE  symbol IN ('AAPL','MSFT','GOOGL','AMZN','META','NVDA','TSLA')
  AND  date >= '2023-01-01'
ORDER  BY symbol, date

In [ ]:
mag7['date'] = pd.to_datetime(mag7['date'])

fig, ax = plt.subplots(figsize=(14, 6))
for sym, grp in mag7.groupby('symbol'):
    norm = grp.set_index('date')['Close'] / grp['Close'].iloc[0] * 100
    ax.plot(norm, label=sym, linewidth=1.5)

ax.set_title('MAG7 — Normalised Price (base = 100 at 2023-01-01)', fontsize=13)
ax.set_ylabel('Indexed Price')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.legend(ncol=4)
plt.tight_layout()
plt.show()

---
## 5 · Price + SMA-20/33 chart for any symbol

In [ ]:
%%sql aapl <<
SELECT date, Close, sma_20, sma_33
FROM   daily
WHERE  symbol = 'AAPL'
  AND  date  >= '2022-01-01'
ORDER  BY date

In [ ]:
aapl['date'] = pd.to_datetime(aapl['date'])
df = aapl.set_index('date')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df['Close'],  color='steelblue', lw=1.2, label='Close')
ax.plot(df['sma_20'], color='orange',    lw=1.5, linestyle='--', label='SMA-20')
ax.plot(df['sma_33'], color='crimson',   lw=1.5, linestyle='--', label='SMA-33')
ax.set_title('AAPL — Close vs SMA-20 / SMA-33 (daily)')
ax.set_ylabel('Price (USD)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.legend()
plt.tight_layout()
plt.show()

---
## 6 · Weekly chart — Close + SMA-20/33

In [ ]:
%%sql wsma <<
SELECT week_start, symbol, Close, sma_20, sma_33
FROM   weekly
WHERE  symbol IN ('AAPL','MSFT','TSLA')
  AND  week_start >= '2020-01-01'
ORDER  BY symbol, week_start

In [ ]:
wsma['week_start'] = pd.to_datetime(wsma['week_start'])
symbols = wsma['symbol'].unique()

fig, axes = plt.subplots(len(symbols), 1, figsize=(14, 4 * len(symbols)), sharex=True)
axes = [axes] if len(symbols) == 1 else axes

for ax, sym in zip(axes, symbols):
    g = wsma[wsma['symbol'] == sym].set_index('week_start')
    ax.plot(g['Close'],  color='steelblue', lw=1,   label='Weekly Close')
    ax.plot(g['sma_20'], color='orange',    lw=1.5, linestyle='--', label='SMA-20w')
    ax.plot(g['sma_33'], color='crimson',   lw=1.5, linestyle='--', label='SMA-33w')
    ax.set_title(f'{sym} — Weekly Close + SMA-20 / SMA-33')
    ax.set_ylabel('Price')
    ax.legend(fontsize=9)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

---
## 7 · Daily return correlation heatmap

In [ ]:
%%sql ret_raw <<
SELECT date, symbol, Close
FROM   daily
WHERE  symbol IN ('AAPL','MSFT','GOOGL','AMZN','META','NVDA','TSLA')
  AND  date >= '2022-01-01'
ORDER  BY symbol, date

In [ ]:
ret_raw['date'] = pd.to_datetime(ret_raw['date'])
pivot = ret_raw.pivot(index='date', columns='symbol', values='Close')
corr  = pivot.pct_change().dropna().corr()
syms  = corr.columns.tolist()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(syms))); ax.set_xticklabels(syms, rotation=45)
ax.set_yticks(range(len(syms))); ax.set_yticklabels(syms)
for i in range(len(syms)):
    for j in range(len(syms)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=9)
ax.set_title('Daily Return Correlation — MAG7 (2022–present)')
plt.tight_layout()
plt.show()

---
## 8 · 52-week high / low screen

In [ ]:
%%sql
WITH w AS (
    SELECT symbol, date, Close,
           MAX(High) OVER (PARTITION BY symbol ORDER BY date
                           ROWS BETWEEN 251 PRECEDING AND CURRENT ROW) AS high_52w,
           MIN(Low)  OVER (PARTITION BY symbol ORDER BY date
                           ROWS BETWEEN 251 PRECEDING AND CURRENT ROW) AS low_52w
    FROM daily
)
SELECT symbol,
       ROUND(Close,    2) AS last_close,
       ROUND(high_52w, 2) AS high_52w,
       ROUND(low_52w,  2) AS low_52w,
       ROUND((Close - low_52w) / NULLIF(high_52w - low_52w, 0) * 100, 1) AS pct_of_range
FROM  w
WHERE date = (SELECT MAX(date) FROM daily)
  AND symbol IN ('AAPL','MSFT','GOOGL','AMZN','META','NVDA','TSLA',
                 'CRWD','NFLX','AMD','QCOM','INTC','ORCL','ADBE')
ORDER BY pct_of_range DESC

---
## 9 · Average annual volume + close

In [ ]:
%%sql
SELECT YEAR(date)              AS yr,
       ROUND(AVG(Volume)/1e6, 1) AS avg_volume_M,
       ROUND(AVG(Close), 2)    AS avg_close
FROM   daily
WHERE  symbol = 'AAPL'
GROUP  BY yr
ORDER  BY yr

---
## 10 · Your own SQL

Edit the query freely.  
Use `%%sql` to display results, or `%%sql myvar <<` to assign to a variable for charting.

In [ ]:
%%sql
SELECT symbol, date,
       ROUND(Close,  2) AS Close,
       ROUND(sma_20, 2) AS sma_20,
       ROUND(sma_33, 2) AS sma_33,
       ROUND(ema_20, 2) AS ema_20,
       ROUND(ema_33, 2) AS ema_33
FROM   daily
WHERE  symbol = 'NVDA'
  AND  date  >= '2024-01-01'
ORDER  BY date DESC
LIMIT  30

In [16]:
%%sql  
SELECT *
FROM   weekly
-- WHERE  symbol IN ('AAPL','MSFT','TSLA')
  where sma_20 is not null

,week_start,symbol,Open,High,Low,Close,Volume,trading_days,sma_20,sma_33,ema_20,ema_33
0,2025-09-08,CTRI,21.7400,23.3090,21.0000,23.0200,15542100,5,20.9625,19.8164,20.7935,20.4655
1,2025-09-29,CTRI,21.2700,22.3200,20.0700,20.3000,11702800,5,21.1275,19.7448,20.8171,20.5375
2,2025-12-08,CTRI,25.8800,27.5000,25.1000,26.5800,11647700,5,21.4025,21.2061,21.7684,21.2482
3,2025-12-22,CTRI,26.3100,26.5300,25.7600,25.9400,2887600,4,21.9550,21.5742,22.5415,21.7945
4,2025-07-07,CTRI,22.1900,22.2700,20.8350,20.9400,3505400,5,19.0260,19.7752,20.1919,19.9688
...,...,...,...,...,...,...,...,...,...,...,...,...
2806555,2022-12-26,CCI,114.6009,116.1419,112.1387,113.5960,5463000,4,120.4846,129.9280,120.3282,126.1224
2806556,2023-05-01,CCI,103.9970,104.5225,98.9705,100.4454,12150700,5,113.8272,113.6738,112.2406,116.6158
2806557,2023-07-17,CCI,98.8164,99.2804,88.6942,92.1828,23792000,5,101.9454,108.1582,101.8389,106.9170
2806558,2023-08-21,CCI,85.9789,86.8639,84.1572,85.6695,13516200,5,96.4716,104.0757,96.5832,102.1367
